# SWDB - Open Source Identification

In [ ]:
# helper tools


In [25]:
import pandas as pd

os_swdb = pd.read_csv('data/open_source_classification.csv')
os_swdb = os_swdb[os_swdb['IsOpenSource'] == 'Yes']

def extract_github_info(row):
    if pd.notna(row['GitHubLink']):
        link = row['GitHubLink']
        if link.startswith('https://github.com/'):
            parts = link[len('https://github.com/'):].strip('/').split('/')
            row['gh_profile_name'] = parts[0]
            row['profile_url'] = f"https://github.com/{parts[0]}"
            if len(parts) >= 2:
                row['gh_repo_name'] = parts[1]
                row['repo_url'] = f"https://github.com/{parts[0]}/{parts[1]}"
    return row

os_swdb = os_swdb.apply(extract_github_info, axis=1)
print(os_swdb.head())

# number of repo_url
print(f"Number of GitHub repositories identified: {os_swdb['repo_url'].nunique()}")

# save in MAIN.csv
os_swdb.to_csv('MAIN.csv', index=False)


                               GitHubLink IsOpenSource                       Product ProductCategory              VendorName gh_profile_name gh_repo_name                    profile_url                              repo_url
490  https://github.com/vdukhovni/postfix          Yes                   Mail Server  Email Software                 POSTFIX       vdukhovni      postfix   https://github.com/vdukhovni  https://github.com/vdukhovni/postfix
537             https://github.com/Zimbra          Yes    Zimbra Email Collaboration  Email Software                 Synacor          Zimbra          NaN      https://github.com/Zimbra                                   NaN
542         https://github.com/ProtonMail          Yes                    ProtonMail  Email Software  Proton Technologies AG      ProtonMail          NaN  https://github.com/ProtonMail                                   NaN
565          https://github.com/Exim/exim          Yes                   Mail Server  Email Software        

# Merge found OSS Projects with SWDB Technologies install universe

In [26]:
# snippet: /Users/dmk6603/Documents/swdb_opensource/1-indentify_open_source/data/swdb_universe_installs.csv
# VendorName,Product,ProductId,TabKeyNewId,ProductSeries,ProductCategory,Total Sites,US Sites,Total Enterprises,Enterprises in US
# AppNexus,AppNexus,4611,4907,Advertising,Ad Exchanges,"696,761","471,708","257,895","130,146"
# AppNexus,AppNexus,4611,4907,Advertising,Ad Exchanges,"696,761","471,708","257,895","130,146"

swdb_universe = pd.read_csv('data/swdb_universe_installs.csv')
# merge swdb_universe with os_swdb on product, vendor and category
MAIN_df = pd.merge(os_swdb, swdb_universe, left_on=['VendorName', 'Product', 'ProductCategory'], right_on=['VendorName', 'Product', 'ProductCategory'], how='left')
# remove duplicates
MAIN_df = MAIN_df.drop_duplicates(subset=['VendorName', 'Product', 'ProductCategory'])
# print number of rows in MAIN_df
print(f"Number of rows in MAIN_df: {len(MAIN_df)}")
# numer of rows with repo
print(f"Number of rows with repo_url: {MAIN_df['repo_url'].notna().sum()}")

# remove rows with repo_url empty
MAIN_df = MAIN_df[MAIN_df['repo_url'].notna()]

for col in ['Total Sites', 'US Sites', 'Total Enterprises', 'Enterprises in US']:
    MAIN_df[col] = MAIN_df[col].astype(str).str.replace(',', '').astype(int)

# keep only one repo_url and sum all installs for the same repo_url
MAIN_df = MAIN_df.groupby('repo_url').agg({
    'VendorName': 'first',
    'Product': 'first',
    'ProductCategory': 'first',
    'Total Sites': 'sum',
    'US Sites': 'sum',
    'Total Enterprises': 'sum',
    'Enterprises in US': 'sum'
}).reset_index()

print(MAIN_df.head())

# print number of rows in MAIN_df
print(f"Number of rows in MAIN_df after grouping: {len(MAIN_df)}")
# save in MAIN.csv
MAIN_df.to_csv('MAIN.csv', index=False)

Number of rows in MAIN_df: 460
Number of rows with repo_url: 383
                                 repo_url            VendorName    Product                 ProductCategory  Total Sites  US Sites  Total Enterprises  Enterprises in US
0      https://github.com/Alluxio/alluxio          Alluxio Inc.    Alluxio             Big Data Processing         9096      4153               2152                832
1       https://github.com/Ametys/runtime                Ametys     Ametys  Web Content Management Systems          535         0                311                  0
2   https://github.com/Automattic/jetpack               Jetpack    JetPack       Other Web Tools & Plugins       286504    158942             179352              78568
3  https://github.com/Automattic/liveblog             WordPress   Liveblog   Application Development Tools            4         1                  4                  1
4  https://github.com/CONTENIDO/CONTENIDO  four for business AG  CONTENIDO  Web Content Managem